# 03 · Days at or above 90°F

**Objective.** Extract the observed 1981–2010 baseline and the projected range of days per year with a
maximum temperature at or above 90°F for the 2030s, 2050s and 2080s from the New York City Panel on Climate
Change (NPCC) projections published by the Mayor's Office of Climate and Environmental Justice.

Input: `../inputs/npcc_extreme_events_projections.csv` (NYC Open Data 38ps-fnsg). The dataset describes its
projections as based on those developed for the IPCC Sixth Assessment Report (AR6); its CMIP6/SSP statement refers
specifically to the sea-level projections.
Outputs: `data/days_at_or_above_90f.json`, `data/days_at_or_above_90f.csv`.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from its own directory (run_notebooks.py does)
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)

import pandas as pd

from common import INPUTS, SOURCES, write_json

COLUMN = "Number of days/year with maximum temperature at or above 90°F"
PERCENTILES = [10, 25, 75, 90]                        # the four percentiles the source publishes; no median is published
PERIODS = [("2030s", 2035), ("2050s", 2055), ("2080s", 2085)]   # horizontal plotting positions: decade midpoints


## 1. Read the baseline and percentile rows

The source publishes one observed baseline value and four percentiles per future period. Each label must match exactly one row and the percentiles
must be ordered. The baseline value is repeated in every percentile field of the baseline row so a plotted
range can start from it; the baseline itself has no model spread.

In [2]:
raw = pd.read_csv(INPUTS / "npcc_extreme_events_projections.csv")
raw[["Period", COLUMN]]


,Period,Number of days/year with maximum temperature at or above 90°F
0,Baseline (1981-2010),17.0
1,2030s (10th Percentile),27.0
2,2030s (25th Percentile),27.0
3,2030s (75th Percentile),46.0
4,2030s (90th Percentile),54.0
5,2050s (10th Percentile),32.0
6,2050s (25th Percentile),38.0
7,2050s (75th Percentile),62.0
8,2050s (90th Percentile),69.0
9,2080s (10th Percentile),46.0


In [3]:
def one(mask: pd.Series, what: str) -> float:
    """Return the single value selected by `mask`; fail if the label matched zero or several rows."""
    matches = raw.loc[mask, COLUMN]
    assert len(matches) == 1, f"{what}: expected exactly one row, found {len(matches)}"
    return float(matches.iloc[0])


baseline = one(raw.Period == "Baseline (1981-2010)", "baseline")
assert baseline == 17, baseline
series = [{"period": "Baseline (1981–2010)", "year": 1995, "observed": True, **{f"p{q}": baseline for q in PERCENTILES}}]
for decade, year in PERIODS:
    row = {"period": decade, "year": year, "observed": False}
    for q in PERCENTILES:
        row[f"p{q}"] = one(raw.Period == f"{decade} ({q}th Percentile)", f"{decade} p{q}")
    assert row["p10"] <= row["p25"] <= row["p75"] <= row["p90"], row
    series.append(row)
pd.DataFrame(series)


,period,year,observed,p10,p25,p75,p90
0,Baseline (1981–2010),1995,True,17.0,17.0,17.0,17.0
1,2030s,2035,False,27.0,27.0,46.0,54.0
2,2050s,2055,False,32.0,38.0,62.0,69.0
3,2080s,2085,False,46.0,46.0,85.0,108.0


## 2. Write the outputs

The chart draws the 25th and 75th percentiles as lines, shades 25th–75th darker and 10th–90th lighter, and
connects periods with straight segments as a visual guide. Percentile ranges describe model spread, not
confidence intervals. The 90°F threshold is the one reported here; heat causes illness and death at lower
temperatures as well.

In [4]:
write_json(DATA / "days_at_or_above_90f.json", {
    "description": "Days per year with maximum air temperature at or above 90°F in New York City: observed 1981–2010 baseline and model percentiles (10th, 25th, 75th, 90th) for the 2030s, 2050s and 2080s. Projections are the NPCC's, based on those developed for the IPCC Sixth Assessment Report. The baseline row repeats the observed value in every percentile field so a plotted range can start from it; the baseline has no model spread.",
    "sources": [SOURCES["npcc"]],
    "threshold_f": 90,
    "units": "days per year",
    "percentiles": PERCENTILES,
    "series": series,
    "plotting_note": "year values are display positions (baseline midpoint 1995; decade midpoints 2035, 2055, 2085). Percentile ranges describe model spread, not confidence intervals.",
})
pd.DataFrame(series).to_csv(DATA / "days_at_or_above_90f.csv", index=False)
print("wrote data/days_at_or_above_90f.csv")


wrote 03_days_at_or_above_90f/data/days_at_or_above_90f.json
wrote data/days_at_or_above_90f.csv
